# Boundary Interpolation

Interpolate gaps of low confidence joints at the movement boundaries

### Imports

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

### Config

In [2]:
CROPPED_DIR    = Path('../data/processed/keypoints_cropped')
DOWN_DIR       = Path('../data/processed/keypoints_cropped_down')
FULL_DIR       = Path('../data/processed/keypoints')
VIDEO_DIR      = Path('../data/Utvalda filminspelningar för IRAF analys/Dec 2025 sit-stå och stå-sitt')
OUT_CSV_BASE   = Path('../data/processed/keypoints_interpolated')
DOWN_OUT_CSV_BASE = Path('../data/processed/keypoints_interpolated_down')
OUT_VIDEO_BASE = Path('../data/processed/keypoints_interpolated_boundary_videos')
DOWN_OUT_VIDEO_BASE = Path('../data/processed/keypoints_interpolated_boundary_videos_down')

THRESHOLD      = 0.6
MAX_INTERP_GAP = 10
FPS            = 50

MODELS = [
    ('movenet',       'confidence'),
    ('mediapipe_norm','visibility'),
]

JOINTS = [
    'nose',
    'left_ear',
    'left_shoulder', 'right_shoulder',
    'left_elbow',    'right_elbow',
    'left_wrist',    'right_wrist',
    'left_hip',      'right_hip',
    'left_knee',     'right_knee',
    'left_ankle',    'right_ankle',
]

MEDIAPIPE_EXTRA_JOINTS = [
    'left_heel',       'right_heel',
    'left_foot_index', 'right_foot_index',
]

def joints_for(model):
    if model == 'mediapipe_norm':
        return JOINTS + MEDIAPIPE_EXTRA_JOINTS
    return JOINTS

MODEL_COORDS = {
    'movenet': ('x', 'y'),
    'mediapipe_norm': ('x', 'y', 'z'),
}

time_dict = {
    '028': (9.90,  11.77), '030': (5.80,  7.63),
    '045': (6.54,   7.98), '047': (7.03,  8.91),
    '059': (6.65,   9.11), '061': (6.00,  9.00),
    '074': (6.07,   7.27), '076': (5.00,  8.00),
    '091': (6.39,   7.61), '093': (5.86,  7.67),
}

stand_to_sit_time_dict = {
    '028': (11.87, 12.87), '030': (8.07,  11.56),
    '045': (7.94,   9.26), '047': (9.20,  12.30),
    '059': (9.11,  11.06), '061': (10.57, 13.41),
    '074': (7.27,   8.20), '076': (8.27,  10.84),
    '091': (7.56,   8.18), '093': (8.50,  11.19),
}

OUT_CSV_BOUNDARY_BASE = OUT_CSV_BASE.parent / f"{OUT_CSV_BASE.name}_boundary"
OUT_CSV_BOUNDARY_BASE.mkdir(parents=True, exist_ok=True)

DOWN_OUT_CSV_BOUNDARY_BASE = DOWN_OUT_CSV_BASE.parent / f"{DOWN_OUT_CSV_BASE.name}_boundary"
DOWN_OUT_CSV_BOUNDARY_BASE.mkdir(parents=True, exist_ok=True)

OUT_CSV_BOUNDARY_BASE, DOWN_OUT_CSV_BOUNDARY_BASE

(PosixPath('../data/processed/keypoints_interpolated_boundary'),
 PosixPath('../data/processed/keypoints_interpolated_down_boundary'))

### Helpers

In [3]:
def good_mask_metric(df, metric_col, threshold):
    v = df[metric_col].to_numpy()
    return (~np.isnan(v)) & (v >= threshold)

def get_frame_to_index(frames):
    return {int(f): i for i, f in enumerate(frames)}

def last_good_before(good_frames, frame):
    b = good_frames[good_frames < frame]
    return int(b[-1]) if len(b) else None

def first_good_after(good_frames, frame):
    a = good_frames[good_frames > frame]
    return int(a[0]) if len(a) else None

def missing_mask_joint(df, joint, coords):
    masks = [df[f'{joint}_{c}'].isna().to_numpy() for c in coords]
    return np.logical_or.reduce(masks)

def leading_missing_end_idx(missing_mask):
    if not missing_mask[0]:
        return None
    good = np.flatnonzero(~missing_mask)
    return (int(good[0]) - 1) if len(good) else (len(missing_mask) - 1)

def trailing_missing_start_idx(missing_mask):
    if not missing_mask[-1]:
        return None
    good = np.flatnonzero(~missing_mask)
    return (int(good[-1]) + 1) if len(good) else 0

def interpolate_frames_between_anchors(df_out, frames_crop, joint, coords,
                                       anchor_before_frame, anchor_after_frame,
                                       full_df, full_frame_to_idx, target_mask):
    i0 = full_frame_to_idx[anchor_before_frame]
    i1 = full_frame_to_idx[anchor_after_frame]

    gap_len = int(anchor_after_frame - anchor_before_frame - 1)
    if gap_len <= 0:
        return

    tgt_frames = frames_crop[target_mask].astype(int)
    k = tgt_frames - anchor_before_frame

    for c in coords:
        col = f'{joint}_{c}'
        v0 = float(full_df.at[i0, col])
        v1 = float(full_df.at[i1, col])
        seq = np.linspace(v0, v1, gap_len + 2)[1:-1]
        df_out.loc[target_mask, col] = seq[k - 1]

### Run

In [4]:
for in_csv_base, out_boundary_base, label in [
    (OUT_CSV_BASE, OUT_CSV_BOUNDARY_BASE, 'sit-to-stand'),
    (DOWN_OUT_CSV_BASE, DOWN_OUT_CSV_BOUNDARY_BASE, 'stand-to-sit'),
]:
    print(f'\n=== {label} ===')
    for model, metric in MODELS:
        coords = MODEL_COORDS[model]

        in_interp_dir = in_csv_base / model
        in_full_dir   = FULL_DIR / model
        out_dir       = out_boundary_base / model
        out_dir.mkdir(parents=True, exist_ok=True)

        files = sorted(in_interp_dir.glob('*.csv'))

        for csv_path in files:
            name = csv_path.name

            df_crop = pd.read_csv(csv_path)
            df_full = pd.read_csv(in_full_dir / name)

            frames_crop = df_crop['frame'].to_numpy().astype(int)
            first_frame = int(frames_crop[0])
            last_frame  = int(frames_crop[-1])

            frames_full = df_full['frame'].to_numpy().astype(int)
            full_frame_to_idx = get_frame_to_index(frames_full)

            df_out = df_crop.copy()

            for joint in joints_for(model):
                metric_col = f'{joint}_{metric}'
                if metric_col not in df_full.columns:
                    continue

                flag_col = f'{joint}_interpolated'
                if flag_col not in df_out.columns:
                    df_out[flag_col] = False

                good_frames_full = frames_full[good_mask_metric(df_full, metric_col, THRESHOLD)]

                miss = missing_mask_joint(df_out, joint, coords)

                lead_end = leading_missing_end_idx(miss)
                if lead_end is not None:
                    anchor_before = last_good_before(good_frames_full, first_frame)

                    after_inside_idx = lead_end + 1
                    anchor_after = None
                    if after_inside_idx < len(frames_crop):
                        f = int(frames_crop[after_inside_idx])
                        if f in full_frame_to_idx:
                            v = df_full.at[full_frame_to_idx[f], metric_col]
                            if (not pd.isna(v)) and (v >= THRESHOLD):
                                anchor_after = f
                    if anchor_after is None:
                        anchor_after = first_good_after(good_frames_full, last_frame)

                    if anchor_before is not None and anchor_after is not None:
                        gap = int(anchor_after - anchor_before - 1)
                        if gap <= MAX_INTERP_GAP:
                            mask = (frames_crop >= first_frame) & (frames_crop <= int(frames_crop[lead_end]))
                            interpolate_frames_between_anchors(
                                df_out, frames_crop, joint, coords,
                                anchor_before, anchor_after,
                                df_full, full_frame_to_idx,
                                mask
                            )
                            df_out.loc[mask, flag_col] = True

                miss = missing_mask_joint(df_out, joint, coords)

                trail_start = trailing_missing_start_idx(miss)
                if trail_start is not None:
                    anchor_after = first_good_after(good_frames_full, last_frame)

                    before_inside_idx = trail_start - 1
                    anchor_before = None
                    if before_inside_idx >= 0:
                        f = int(frames_crop[before_inside_idx])
                        if f in full_frame_to_idx:
                            v = df_full.at[full_frame_to_idx[f], metric_col]
                            if (not pd.isna(v)) and (v >= THRESHOLD):
                                anchor_before = f
                    if anchor_before is None:
                        anchor_before = last_good_before(good_frames_full, first_frame)

                    if anchor_before is not None and anchor_after is not None:
                        gap = int(anchor_after - anchor_before - 1)
                        if gap <= MAX_INTERP_GAP:
                            mask = (frames_crop >= int(frames_crop[trail_start])) & (frames_crop <= last_frame)
                            interpolate_frames_between_anchors(
                                df_out, frames_crop, joint, coords,
                                anchor_before, anchor_after,
                                df_full, full_frame_to_idx,
                                mask
                            )
                            df_out.loc[mask, flag_col] = True

            df_out.to_csv(out_dir / name, index=False)

        print(f'{model}: wrote {len(files)} files to {out_dir}')


=== sit-to-stand ===
movenet: wrote 10 files to ../data/processed/keypoints_interpolated_boundary/movenet
mediapipe_norm: wrote 10 files to ../data/processed/keypoints_interpolated_boundary/mediapipe_norm

=== stand-to-sit ===
movenet: wrote 10 files to ../data/processed/keypoints_interpolated_down_boundary/movenet
mediapipe_norm: wrote 10 files to ../data/processed/keypoints_interpolated_down_boundary/mediapipe_norm


### Summary

In [5]:
summary = []

for in_csv_base, out_boundary_base, label in [
    (OUT_CSV_BASE, OUT_CSV_BOUNDARY_BASE, 'sit-to-stand'),
    (DOWN_OUT_CSV_BASE, DOWN_OUT_CSV_BOUNDARY_BASE, 'stand-to-sit'),
]:
    for model, _ in MODELS:
        before_dir = in_csv_base / model
        after_dir  = out_boundary_base / model

        total_coord_changes = 0
        total_flag_changes  = 0
        total_files         = 0

        for csv_path in sorted(before_dir.glob("*.csv")):
            name = csv_path.name
            df_before = pd.read_csv(csv_path)
            df_after  = pd.read_csv(after_dir / name)

            coord_cols = [c for c in df_before.columns if c.endswith(('_x','_y','_z'))]
            flag_cols  = [c for c in df_before.columns if c.endswith('_interpolated')]

            coord_diff = (df_before[coord_cols].fillna(0) != df_after[coord_cols].fillna(0)).sum().sum()
            flag_diff  = (df_before[flag_cols] != df_after[flag_cols]).sum().sum()

            total_coord_changes += coord_diff
            total_flag_changes  += flag_diff
            total_files += 1

        summary.append({
            "label": label,
            "model": model,
            "files": total_files,
            "coord_values_changed": int(total_coord_changes),
            "flags_changed": int(total_flag_changes),
        })

pd.DataFrame(summary)

,label,model,files,coord_values_changed,flags_changed
0,sit-to-stand,movenet,10,104,52
1,sit-to-stand,mediapipe_norm,10,0,0
2,stand-to-sit,movenet,10,92,46
3,stand-to-sit,mediapipe_norm,10,18,6


In [6]:
filled_gaps_all = []

for in_csv_base, out_boundary_base, label in [
    (OUT_CSV_BASE, OUT_CSV_BOUNDARY_BASE, 'sit-to-stand'),
    (DOWN_OUT_CSV_BASE, DOWN_OUT_CSV_BOUNDARY_BASE, 'stand-to-sit'),
]:
    for model, _ in MODELS:
        before_dir = in_csv_base / model
        after_dir  = out_boundary_base / model
        coords = MODEL_COORDS[model]

        for csv_path in sorted(before_dir.glob("*.csv")):
            name = csv_path.name
            df_before = pd.read_csv(csv_path)
            df_after  = pd.read_csv(after_dir / name)

            frames = df_before["frame"].to_numpy().astype(int)
            n = len(frames)
            first_frame = int(frames[0])
            last_frame  = int(frames[-1])

            for joint in joints_for(model):
                coord_cols = [f"{joint}_{c}" for c in coords if f"{joint}_{c}" in df_before.columns]
                if not coord_cols:
                    continue

                was_nan = df_before[coord_cols].isna().any(axis=1).to_numpy()
                now_nan = df_after[coord_cols].isna().any(axis=1).to_numpy()

                filled = was_nan & (~now_nan)
                if not filled.any():
                    continue

                padded = np.pad(filled, (1, 1), constant_values=False)
                diff = np.diff(padded.astype(int))
                starts = np.where(diff == 1)[0]
                ends   = np.where(diff == -1)[0]

                for s, e in zip(starts, ends):
                    start_idx = int(s)
                    end_idx_excl = int(e)
                    start_frame = int(frames[start_idx])
                    end_frame   = int(frames[end_idx_excl - 1])
                    length = int(end_idx_excl - start_idx)

                    touches_start = (start_idx == 0)
                    touches_end   = (end_idx_excl == n)

                    filled_gaps_all.append({
                        "label": label,
                        "model": model,
                        "file": name,
                        "joint": joint,
                        "start_frame": start_frame,
                        "end_frame": end_frame,
                        "length_frames": length,
                        "touches_start": touches_start,
                        "touches_end": touches_end,
                        "crop_first_frame": first_frame,
                        "crop_last_frame": last_frame,
                    })

filled_gaps_df = pd.DataFrame(filled_gaps_all)
filled_gaps_df

,label,model,file,joint,start_frame,end_frame,length_frames,touches_start,touches_end,crop_first_frame,crop_last_frame
0,sit-to-stand,movenet,DJI_20250425092743_0028_D_movenet.csv,left_elbow,495,497,3,True,False,495,588
1,sit-to-stand,movenet,DJI_20250425093100_0030_D_movenet.csv,left_shoulder,290,290,1,True,False,290,381
2,sit-to-stand,movenet,DJI_20250425093100_0030_D_movenet.csv,right_elbow,290,290,1,True,False,290,381
3,sit-to-stand,movenet,DJI_20250425093100_0030_D_movenet.csv,left_wrist,290,290,1,True,False,290,381
4,sit-to-stand,movenet,DJI_20250425093100_0030_D_movenet.csv,right_wrist,381,381,1,False,True,290,381
5,sit-to-stand,movenet,DJI_20250425093100_0030_D_movenet.csv,right_hip,290,295,6,True,False,290,381
6,sit-to-stand,movenet,DJI_20250425093100_0030_D_movenet.csv,right_ankle,290,290,1,True,False,290,381
7,sit-to-stand,movenet,DJI_20250425104507_0045_D_movenet.csv,nose,398,399,2,False,True,327,399
8,sit-to-stand,movenet,DJI_20250425104507_0045_D_movenet.csv,left_elbow,327,327,1,True,False,327,399
9,sit-to-stand,movenet,DJI_20250425104507_0045_D_movenet.csv,right_elbow,398,399,2,False,True,327,399


In [7]:
if len(filled_gaps_df) == 0:
    print("No filled gaps detected.")
else:
    filled_gaps_df["is_boundary_run"] = filled_gaps_df["touches_start"] | filled_gaps_df["touches_end"]

    display(filled_gaps_df["is_boundary_run"].value_counts().to_frame("count"))

    non_boundary = filled_gaps_df[~filled_gaps_df["is_boundary_run"]]
    if len(non_boundary):
        print("WARNING: Found filled runs that do NOT touch a boundary (unexpected).")
        display(non_boundary.sort_values(["model","file","joint","start_frame"]).head(50))
    else:
        print("All filled runs touch a boundary (as expected).")

,count
is_boundary_run,
True,53


All filled runs touch a boundary (as expected).


## Interpolate Z

* if xy is NaN, z should also be NaN
* if xy is interpolated, z should also be interpolated

In [8]:
for out_boundary_base in [OUT_CSV_BOUNDARY_BASE, DOWN_OUT_CSV_BOUNDARY_BASE]:
    IN_DIR = out_boundary_base / 'mediapipe_norm'

    for csv_path in sorted(IN_DIR.glob('*.csv')):
        df = pd.read_csv(csv_path).reset_index(drop=True)
        n = len(df)

        for joint in joints_for('mediapipe_norm'):
            flag = f'{joint}_interpolated'
            xcol = f'{joint}_x'
            ycol = f'{joint}_y'
            zcol = f'{joint}_z'
            if flag not in df.columns or xcol not in df.columns or ycol not in df.columns or zcol not in df.columns:
                continue

            interp = df[flag].to_numpy().astype(bool)

            padded = np.pad(interp, (1, 1), constant_values=False)
            diff = np.diff(padded.astype(int))
            starts = np.where(diff == 1)[0]
            ends   = np.where(diff == -1)[0]

            for s, e in zip(starts, ends):
                start = int(s)
                end   = int(e)

                if start == 0 or end == n:
                    continue

                z0 = df.at[start - 1, zcol]
                z1 = df.at[end, zcol]
                if pd.isna(z0) or pd.isna(z1):
                    continue

                gap_len = end - start
                df.loc[start:end - 1, zcol] = np.linspace(z0, z1, gap_len + 2)[1:-1]

            xy_missing = df[xcol].isna() | df[ycol].isna()
            df.loc[xy_missing, zcol] = np.nan

        df.to_csv(csv_path, index=False)

    print("MediaPipe z fixed in place (interior runs only):", IN_DIR)

MediaPipe z fixed in place (interior runs only): ../data/processed/keypoints_interpolated_boundary/mediapipe_norm
MediaPipe z fixed in place (interior runs only): ../data/processed/keypoints_interpolated_down_boundary/mediapipe_norm


### Summary

In [9]:
for in_csv_base, out_boundary_base, label in [
    (OUT_CSV_BASE, OUT_CSV_BOUNDARY_BASE, 'sit-to-stand'),
    (DOWN_OUT_CSV_BASE, DOWN_OUT_CSV_BOUNDARY_BASE, 'stand-to-sit'),
]:
    print(f'\n=== {label} ===')
    before_dir = in_csv_base / "mediapipe_norm"
    after_dir  = out_boundary_base / "mediapipe_norm"
    rows = []

    for p in sorted(before_dir.glob("*.csv")):
        name = p.name
        df_b = pd.read_csv(p)
        df_a = pd.read_csv(after_dir / name)

        frames = df_b["frame"].to_numpy().astype(int)
        n = len(frames)

        for joint in joints_for('mediapipe_norm'):
            flag = f"{joint}_interpolated"
            xcol = f"{joint}_x"
            ycol = f"{joint}_y"
            zcol = f"{joint}_z"
            if flag not in df_a.columns or zcol not in df_b.columns or zcol not in df_a.columns:
                continue
            if xcol not in df_a.columns or ycol not in df_a.columns:
                continue

            z_before = df_b[zcol]
            z_after  = df_a[zcol]

            z_changed = (z_before.fillna(0) != z_after.fillna(0)).to_numpy()
            if not z_changed.any():
                continue

            padded = np.pad(z_changed, (1, 1), constant_values=False)
            diff = np.diff(padded.astype(int))
            starts = np.where(diff == 1)[0]
            ends   = np.where(diff == -1)[0]

            interp = df_a[flag].astype(bool).to_numpy()
            xy_missing = (df_a[xcol].isna() | df_a[ycol].isna()).to_numpy()

            for s, e in zip(starts, ends):
                s = int(s); e = int(e)

                touches_start = (s == 0)
                touches_end   = (e == n)
                all_interp    = bool(interp[s:e].all())
                any_interp    = bool(interp[s:e].any())
                all_xy_missing = bool(xy_missing[s:e].all())

                if all_interp:
                    change_type = "z interpolated (xy interpolated)"
                elif all_xy_missing:
                    change_type = "z set to NaN (xy missing)"
                elif any_interp:
                    change_type = "mixed (some interpolated frames)"
                else:
                    change_type = "other (check)"

                rows.append({
                    "file": name,
                    "joint": joint,
                    "start_frame": int(frames[s]),
                    "end_frame": int(frames[e - 1]),
                    "length": int(e - s),
                    "touches_start": touches_start,
                    "touches_end": touches_end,
                    "change_type": change_type,
                })

    analysis_df = pd.DataFrame(rows)

    display(
        analysis_df.groupby("change_type")["length"]
        .agg(runs="count", frames="sum", mean_len="mean", max_len="max")
        .sort_values("frames", ascending=False)
    )


=== sit-to-stand ===


,runs,frames,mean_len,max_len
change_type,,,,
z set to NaN (xy missing),46,1950,42.391304,118
z interpolated (xy interpolated),2,4,2.000000,3



=== stand-to-sit ===


,runs,frames,mean_len,max_len
change_type,,,,
z set to NaN (xy missing),29,1115,38.448276,72
z interpolated (xy interpolated),10,32,3.200000,10


## Video Output

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import imageio

def draw_keypoints_on_image(image, keypoints, model):
    height, width, _ = image.shape
    aspect_ratio = float(width) / height
    fig, ax = plt.subplots(figsize=(12 * aspect_ratio, 12))
    fig.tight_layout(pad=0)
    ax.margins(0)
    ax.set_yticklabels([])
    ax.set_xticklabels([])
    plt.axis('off')

    ax.imshow(image)

    for joint in joints_for(model):
        x = keypoints.get(f"{joint}_x")
        y = keypoints.get(f"{joint}_y")
        if pd.isna(x) or pd.isna(y):
            continue

        interp = bool(keypoints.get(f"{joint}_interpolated", False))
        c = "#00ff00" if not interp else "#ffa500"
        ax.scatter([x * width], [y * height], c=c)

    fig.canvas.draw()
    image_from_plot = np.frombuffer(fig.canvas.tostring_argb(), dtype=np.uint8)
    image_from_plot = image_from_plot.reshape(fig.canvas.get_width_height()[::-1] + (4,))
    image_from_plot = image_from_plot[:, :, 1:4]
    plt.close(fig)

    return image_from_plot


def draw_keypoints_on_video(video_path, keypoints_df, out_path, model):
    reader = imageio.get_reader(str(video_path))
    fps = reader.get_meta_data().get('fps', FPS)

    start_frame = int(keypoints_df['frame'].iloc[0])
    n_frames = len(keypoints_df)

    with imageio.get_writer(str(out_path), fps=fps) as writer:
        for i in range(n_frames):
            frame = reader.get_data(start_frame + i)
            row = keypoints_df.iloc[i]
            image = draw_keypoints_on_image(frame, row, model)
            writer.append_data(image)


for cropped_dir, out_boundary_base, out_video_base, label in [
    (CROPPED_DIR, OUT_CSV_BOUNDARY_BASE, OUT_VIDEO_BASE, 'sit-to-stand'),
    (DOWN_DIR, DOWN_OUT_CSV_BOUNDARY_BASE, DOWN_OUT_VIDEO_BASE, 'stand-to-sit'),
]:
    print(f'\n=== {label} ===')
    for model, metric in MODELS:
        out_video = out_video_base / model
        out_video.mkdir(parents=True, exist_ok=True)

        for csv_path in sorted((cropped_dir / model).glob('*.csv')):
            match = re.search(r'_0(\d{3})_D_', csv_path.stem)
            if not match:
                continue
            vid_id = match.group(1)

            vid_files = list(VIDEO_DIR.glob(f'*_0{vid_id}_D.MP4'))
            if not vid_files:
                print(f'Video not found: {vid_id}')
                continue

            df_interp = pd.read_csv(out_boundary_base / model / csv_path.name).reset_index(drop=True)

            suffix   = f'_{model}'
            out_name = csv_path.stem.replace(suffix, '') + '_interpolated.mp4'
            out_path = out_video / out_name

            draw_keypoints_on_video(vid_files[0], df_interp, out_path, model)
            print(f'{model} {vid_id}: {len(df_interp)} frames -> {out_name}')